# Initialize Store & Generate Tile List

Run this notebook **once** before any tile processing.

1. Reads `config.txt` and shows dataset parameters
2. Computes the land tile list (via cartopy) and saves it to `tile_list.json`
3. Creates the empty Icechunk/Zarr v3 store on Azure Blob Storage

After this, trigger the **Process All Tiles** GitHub Actions workflow to fill the store.

In [ ]:
import json
import sys
from pathlib import Path

import dask.array as da
import icechunk
import numpy as np
import xarray as xr
from cartopy.feature import LAND

sys.path.insert(0, str(Path.cwd().parent))
from utils import get_storage, load_config

cfg = load_config()
YEARS         = cfg["YEARS"]
RESOLUTION    = cfg["RESOLUTION"]
TILE_SIZE_DEG = cfg["TILE_SIZE_DEG"]
PIXELS_PER_TILE = cfg["PIXELS_PER_TILE"]
TILE_ROWS     = cfg["TILE_ROWS"]
TILE_COLS     = cfg["TILE_COLS"]
FILL_VALUE    = cfg["FILL_VALUE"]

print("Config:", cfg)

## 1. Generate and save tile list

Uses cartopy's Natural Earth LAND feature to identify which 10°×10° tiles intersect land.
The result is saved to `tile_list.json` and **committed to the repo** — it is not recomputed during CI runs.

In [ ]:
tile_list_path = Path.cwd().parent / "tile_list.json"

land_tiles = []
for row in range(TILE_ROWS):
    for col in range(TILE_COLS):
        lat_min = -90 + row * TILE_SIZE_DEG
        lon_min = -180 + col * TILE_SIZE_DEG
        extent = (lon_min, lon_min + TILE_SIZE_DEG, lat_min, lat_min + TILE_SIZE_DEG)
        if list(LAND.intersecting_geometries(extent)):
            land_tiles.append({"row": row, "col": col})

tile_list_path.write_text(json.dumps(land_tiles, indent=2))
print(f"{len(land_tiles)} land tiles written to {tile_list_path}")

## 2. Initialize the Icechunk store

Set the four Azure environment variables before running this cell:
```
AZURE_STORAGE_ACCOUNT, AZURE_STORAGE_SAS_TOKEN, AZURE_CONTAINER, ICECHUNK_PREFIX
```

This creates only metadata and coordinates — no data chunks until tile runners fill them.

In [ ]:
n_lat = TILE_ROWS * PIXELS_PER_TILE
n_lon = TILE_COLS * PIXELS_PER_TILE
shape  = (len(YEARS), n_lat, n_lon)
chunks = (1, PIXELS_PER_TILE, PIXELS_PER_TILE)

lats = np.arange(90, -90, -RESOLUTION) - RESOLUTION / 2
lons = np.arange(-180, 180,  RESOLUTION) + RESOLUTION / 2

var_attrs = {
    "scale_factor": np.float32(0.02),
    "add_offset": np.float32(0.0),
    "_FillValue": FILL_VALUE,
    "valid_range": [7500, 65535],
    "units": "K",
    "grid_mapping": "spatial_ref",
}

ds = xr.Dataset(
    {
        "avg_daytime_lst": xr.DataArray(
            da.full(shape, np.uint16(FILL_VALUE), dtype=np.uint16, chunks=chunks),
            dims=["year", "latitude", "longitude"],
            attrs={**var_attrs, "long_name": "Annual mean daytime land surface temperature"},
        ),
        "max_daytime_lst": xr.DataArray(
            da.full(shape, np.uint16(FILL_VALUE), dtype=np.uint16, chunks=chunks),
            dims=["year", "latitude", "longitude"],
            attrs={**var_attrs, "long_name": "Annual maximum daytime land surface temperature"},
        ),
    },
    coords={"year": np.array(YEARS), "latitude": lats, "longitude": lons},
)
ds.attrs = {
    "title": "MODIS MOD11A2 Annual Daytime Land Surface Temperature",
    "source": "MODIS Terra MOD11A2 Version 6.1 via Microsoft Planetary Computer",
    "Conventions": "CF-1.8",
}
print(ds)

In [ ]:
storage = get_storage()
repo = icechunk.Repository.create(storage)
session = repo.writable_session("main")

ds.to_zarr(
    session.store,
    mode="w",
    zarr_format=3,
    compute=False,
    write_empty_chunks=False,
    consolidated=False,
)

snapshot_id = session.commit("initialize store: empty template")
print(f"Store initialized. Snapshot ID: {snapshot_id}")